# 13 — MLflow Agent Evaluation and Quality

So far we have built capabilities: Research Agent, Portfolio Agent, Multi-Agent Orchestrator, governed concentration-risk function, and the `ai_decide` decision-layer concept.

Now the question becomes:

> **How do we know the system is actually behaving correctly?**

For an enterprise agent, a response that simply looks good is not enough. We want repeatable checks for routing, tool selection, required business facts, research citations, and governed risk-policy results.

For this POC we start with **answer-sheet evaluation**: inputs + representative pre-computed outputs + expectations. Later, once the complete orchestrator is packaged as one callable application, the same golden test set can be used with direct `predict_fn` evaluation.


## 1. Install MLflow

MLflow gives us tracing for **what happened** and evaluation for **whether it was good**.

The core of this notebook uses deterministic code-based scorers, which keeps the POC friendly to our Free Edition workspace.


In [0]:
%pip install -q --upgrade "mlflow[databricks]>=3.14.0"
dbutils.library.restartPython()


## 2. Configure the experiment

We keep evaluation runs in one stable MLflow experiment so future prompt, model, routing, or tool changes can be compared against the same tests.


In [0]:
import mlflow

mlflow.set_tracking_uri("databricks")
EXPERIMENT_NAME = "/Shared/investment_ai_agent_evaluation"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow experiment: {EXPERIMENT_NAME}")


## 3. Build a small golden evaluation set

We test four important behaviors:

- research question,
- portfolio analytics question,
- deterministic risk-policy question,
- multi-agent question.

Each record contains `inputs`, representative `outputs`, and `expectations`.

These are **POC answer-sheet examples** for demonstrating MLflow evaluation. Later we can replace the outputs with actual stored traces or directly run the packaged orchestrator.


In [0]:
eval_data = [
    {
        "inputs": {"question": "Why did Synthetic Compute Company 01 lower its full-year outlook?"},
        "outputs": {
            "response": "The company lowered its outlook due to export-license timing, a delayed hyperscale qualification cycle, and a less favorable second-half product mix. [Research citation]",
            "route": "research",
            "tools_used": ["ai_search"],
            "citations": ["Research citation"],
            "risk_status": None,
        },
        "expectations": {
            "expected_route": "research",
            "expected_tools": ["ai_search"],
            "expected_facts": ["export-license", "hyperscale qualification", "product mix"],
            "requires_citation": True,
            "expected_risk_status": None,
        },
    },
    {
        "inputs": {"question": "What are the largest holdings in this portfolio?"},
        "outputs": {
            "response": "The portfolio agent used the governed portfolio semantic layer to identify the largest holdings by market value.",
            "route": "portfolio",
            "tools_used": ["genie_metric_view"],
            "citations": [],
            "risk_status": None,
        },
        "expectations": {
            "expected_route": "portfolio",
            "expected_tools": ["genie_metric_view"],
            "expected_facts": ["largest holdings", "market value"],
            "requires_citation": False,
            "expected_risk_status": None,
        },
    },
    {
        "inputs": {"question": "Does the 15% position in Synthetic Compute Company 01 breach our 10% concentration policy?"},
        "outputs": {
            "response": "Yes. The position is 15%, above the 10% concentration threshold. The governed result is BREACH and the position exceeds the limit by 5%.",
            "route": "risk",
            "tools_used": ["check_portfolio_concentration"],
            "citations": [],
            "risk_status": "BREACH",
        },
        "expectations": {
            "expected_route": "risk",
            "expected_tools": ["check_portfolio_concentration"],
            "expected_facts": ["15%", "10%", "BREACH", "5%"],
            "requires_citation": False,
            "expected_risk_status": "BREACH",
        },
    },
    {
        "inputs": {"question": "Why did Synthetic Compute Company 01 lower guidance, how large is our exposure, and does it breach concentration policy?"},
        "outputs": {
            "response": "Research indicates the outlook was reduced because of export-license timing and delayed hyperscale qualification. Portfolio exposure is 15%. The governed check returns BREACH against the 10% limit, exceeding it by 5%. [Research citation]",
            "route": "multi_agent",
            "tools_used": ["ai_search", "genie_metric_view", "check_portfolio_concentration"],
            "citations": ["Research citation"],
            "risk_status": "BREACH",
        },
        "expectations": {
            "expected_route": "multi_agent",
            "expected_tools": ["ai_search", "genie_metric_view", "check_portfolio_concentration"],
            "expected_facts": ["export-license", "15%", "BREACH", "10%", "5%"],
            "requires_citation": True,
            "expected_risk_status": "BREACH",
        },
    },
]

print(f"Evaluation cases: {len(eval_data)}")
for i, row in enumerate(eval_data, 1):
    print(f"{i}. {row['inputs']['question']}")


## 4. Define deterministic quality scorers

We create four MLflow checks:

- **Routing accuracy** — did we choose the expected capability?
- **Tool-selection accuracy** — did we use the required tools?
- **Required-facts coverage** — did the answer contain the business facts we expect?
- **Governance contract** — are citations present when required, and is the governed risk status preserved?

These are good deterministic checks because we do not need another LLM to decide whether the correct tool was called or whether `BREACH` was preserved.


In [0]:
from typing import Any
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer

@scorer
def routing_accuracy(outputs: dict[str, Any], expectations: dict[str, Any]) -> Feedback:
    actual = outputs.get("route")
    expected = expectations.get("expected_route")
    return Feedback(
        value=actual == expected,
        rationale=f"Expected route={expected}; actual route={actual}",
    )

@scorer
def tool_selection_accuracy(outputs: dict[str, Any], expectations: dict[str, Any]) -> Feedback:
    actual = set(outputs.get("tools_used", []))
    expected = set(expectations.get("expected_tools", []))
    missing = sorted(expected - actual)
    return Feedback(
        value=not missing,
        rationale="All expected tools were used." if not missing else f"Missing expected tools: {missing}",
    )

@scorer
def required_facts_coverage(outputs: dict[str, Any], expectations: dict[str, Any]) -> Feedback:
    response = (outputs.get("response") or "").lower()
    expected_facts = expectations.get("expected_facts", [])
    if not expected_facts:
        return Feedback(value=1.0, rationale="No required facts defined.")
    matched = [f for f in expected_facts if str(f).lower() in response]
    score = len(matched) / len(expected_facts)
    return Feedback(
        value=score,
        rationale=f"Matched {len(matched)}/{len(expected_facts)} required facts: {matched}",
    )

@scorer
def governance_contract(outputs: dict[str, Any], expectations: dict[str, Any]) -> list[Feedback]:
    feedback = []
    requires_citation = expectations.get("requires_citation", False)
    citation_ok = (not requires_citation) or bool(outputs.get("citations", []))
    feedback.append(Feedback(
        name="citation_contract",
        value=citation_ok,
        rationale="Citation requirement satisfied." if citation_ok else "Required citation was missing.",
    ))

    expected_risk = expectations.get("expected_risk_status")
    if expected_risk is not None:
        actual_risk = outputs.get("risk_status")
        feedback.append(Feedback(
            name="risk_policy_contract",
            value=actual_risk == expected_risk,
            rationale=f"Expected risk={expected_risk}; actual={actual_risk}",
        ))
    return feedback


## 5. Run MLflow evaluation

Because `eval_data` already contains outputs, MLflow evaluates it in **answer-sheet mode**. This is useful when we already have agent outputs or stored traces and want to iterate on quality checks without rerunning every agent.


In [0]:
# MLflow logs every expectation entry as an assessment, and assessment values
# cannot be None. Drop None-valued expectations (e.g. expected_risk_status=None
# on non-risk questions) before evaluating; the scorers already treat missing
# keys as "not applicable".
eval_data_clean = [
    {
        **row,
        "expectations": {k: v for k, v in row.get("expectations", {}).items() if v is not None},
    }
    for row in eval_data
]

core_results = mlflow.genai.evaluate(
    data=eval_data_clean,
    scorers=[
        routing_accuracy,
        tool_selection_accuracy,
        required_facts_coverage,
        governance_contract,
    ],
)

print("Evaluation run ID:", core_results.run_id)
print("\nAggregated metrics:")
for name, value in core_results.metrics.items():
    print(f"{name}: {value}")


## 6. Inspect record-level results

Aggregated metrics tell us whether the version looks healthy. Record-level results tell us **why** something failed.

For example, if routing passes but tool selection fails, we know the issue is not the orchestrator's intent classification — it is the execution path after routing.

MLflow also records these assessments in the experiment UI.


In [0]:
display(core_results.result_df)


## 7. Optional: add an LLM judge

Exact contracts are best handled with deterministic scorers, but some quality dimensions are semantic.

For example:

> Is the final response actually relevant to the user's question?

MLflow provides built-in LLM judges such as `RelevanceToQuery`.

Because model-backed judge availability can vary by workspace and Free Edition capability, this section is optional. If it is unavailable, the deterministic evaluation above is still useful and complete for our POC.


In [0]:
try:
    from mlflow.genai.scorers import RelevanceToQuery

    semantic_results = mlflow.genai.evaluate(
        data=eval_data,
        scorers=[RelevanceToQuery()],
    )

    print("✅ Built-in LLM judge evaluation completed.")
    print(semantic_results.metrics)

except Exception as e:
    print("ℹ️ Built-in LLM judge is not available in this workspace.")
    print("The deterministic MLflow evaluation above is still valid.")
    print("\nDatabricks / MLflow message:")
    print(e)


## 8. Production evolution — direct end-to-end evaluation

Today we used answer-sheet evaluation because our capabilities were developed across separate notebooks.

Once the complete application is packaged behind one callable entry point, for example:

```python
run_multi_agent_orchestrator(question)
```

we can use the same golden cases with:

```python
mlflow.genai.evaluate(
    data=golden_dataset,
    predict_fn=run_multi_agent_orchestrator,
    scorers=[...],
)
```

At that point MLflow can run the real application for every test case, capture traces, and evaluate the actual path:

```text
User question
    ↓
Orchestrator routing
    ↓
Agent selection
    ↓
Tool calls
    ↓
AI Search / Genie / UC Function
    ↓
Final response
```

That is the production direction.


# ✅ Notebook 13 complete

We now have a repeatable quality framework around our Investment AI system.

The main lesson is:

> **Do not judge an agent only by whether the final answer sounds good. Evaluate the behavior that produced the answer.**

For our system that means measuring:

```text
Routing
Tool selection
Required business facts
Research citations
Governed risk-policy results
Semantic answer quality
```

MLflow lets us record these evaluation runs and compare future versions so prompt, model, routing, tool, retrieval, or business-logic changes do not silently introduce regressions.

### Next

The next stage is moving toward **production-grade agent architecture**: stronger observability, governed application packaging, monitoring, security, and deployment patterns.
